# 🏦 Predicción de Bancarrota Empresarial con Machine Learning
### Clasificación Binaria con Ratios Financieros

---

**Objetivo:** Construir y entender modelos de Machine Learning que predicen si una empresa entrará en bancarrota, usando únicamente ratios financieros como variables predictoras.

**Variable objetivo:**
- `0` → Empresa NO en bancarrota
- `1` → Empresa EN bancarrota

**Modelos utilizados:**
1. 📈 Regresión Logística
2. 🌳 Árbol de Decisión
3. 🌲 Random Forest *(para comparación de importancia de variables)*

---

> 💡 **Filosofía del proyecto:** No se trata de usar muchos algoritmos, sino de entender profundamente cómo funciona cada uno y qué nos dice sobre el riesgo financiero empresarial.

**Contenido:**
1. Exploración del Dataset
2. Análisis Financiero Conceptual
3. Preprocesamiento de Datos
4. Implementación de Modelos
5. Evaluación de Modelos
6. Interpretabilidad
7. Visualizaciones
8. Conclusiones

## ⚙️ Configuración Inicial
Instalación e importación de todas las librerías necesarias.

In [ ]:
# ============================================================
# LIBRERÍAS
# ============================================================

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Machine Learning - Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Machine Learning - Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Machine Learning - Evaluación
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay
)

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

# Suprimir advertencias menores
import warnings
warnings.filterwarnings('ignore')

print('✅ Todas las librerías importadas correctamente')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   sklearn importado')

---
## 📊 Sección 1: Exploración y Comprensión del Dataset

Antes de aplicar cualquier modelo, necesitamos entender qué datos tenemos. Esta etapa se llama **EDA (Exploratory Data Analysis)** y es fundamental para tomar buenas decisiones más adelante.

> 🔍 **¿Por qué explorar antes de modelar?**  
> Un modelo de ML es tan bueno como los datos que lo alimentan. Si no entendemos los datos, el modelo puede aprender patrones equivocados.

In [ ]:
# ============================================================
# CARGA DEL DATASET
# ------------------------------------------------------------
# Subir el archivo data.csv a Google Colab antes de ejecutar.
# Menú: Archivos (ícono carpeta) → Subir → seleccionar data.csv
# ============================================================

# Si estás en Google Colab y subiste el archivo manualmente:
df = pd.read_csv('data.csv')

# Limpiar espacios en los nombres de columnas (el dataset los tiene)
df.columns = df.columns.str.strip()

print('✅ Dataset cargado exitosamente')
print(f'   Filas    : {df.shape[0]:,}')
print(f'   Columnas : {df.shape[1]}')

In [ ]:
# ============================================================
# VISTA GENERAL DEL DATASET
# ============================================================

print('=' * 60)
print('DIMENSIONES DEL DATASET')
print('=' * 60)
print(f'  Empresas (filas)          : {df.shape[0]:,}')
print(f'  Variables (columnas)      : {df.shape[1]}')
print(f'  Variables predictoras     : {df.shape[1] - 1}')
print(f'  Variable objetivo         : Bankrupt?')
print()

print('=' * 60)
print('PRIMERAS 3 FILAS')
print('=' * 60)
df.head(3)

In [ ]:
# ============================================================
# TIPOS DE DATOS Y VALORES NULOS
# ------------------------------------------------------------
# Verificamos:
#   - Que todos los datos sean numéricos (ratios financieros)
#   - Que no haya valores faltantes que debamos tratar
# ============================================================

print('TIPOS DE DATOS:')
print(df.dtypes.value_counts())
print()

total_nulos = df.isnull().sum().sum()
print(f'VALORES NULOS TOTALES: {total_nulos}')
if total_nulos == 0:
    print('✅ El dataset no tiene valores nulos. No se necesita imputación.')
else:
    print('⚠️  Hay valores nulos que deben tratarse.')
    print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (Bankrupt?)
# ------------------------------------------------------------
# Este es uno de los análisis más importantes:
# Si las clases están muy desbalanceadas, los modelos pueden
# "hacer trampa" prediciendo siempre la clase mayoritaria.
# ============================================================

conteo = df['Bankrupt?'].value_counts()
porcentaje = df['Bankrupt?'].value_counts(normalize=True) * 100

print('=' * 50)
print('DISTRIBUCIÓN DE CLASES')
print('=' * 50)
print(f"  Clase 0 (NO bancarrota) : {conteo[0]:,} empresas  ({porcentaje[0]:.2f}%)")
print(f"  Clase 1 (EN bancarrota) : {conteo[1]:,} empresas  ({porcentaje[1]:.2f}%)")
print()
ratio = conteo[0] / conteo[1]
print(f"  Ratio de desbalance     : {ratio:.0f}:1  (hay {ratio:.0f} empresas sanas por cada 1 en quiebra)")
print()
print('⚠️  DESBALANCE SEVERO: Solo el 3.23% de las empresas están en bancarrota.')
print('   Esto es REALISTA: las quiebras son eventos poco frecuentes.')
print('   Pero requiere que usemos class_weight="balanced" en los modelos.')

In [ ]:
# ============================================================
# VISUALIZACIÓN: DISTRIBUCIÓN DE CLASES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfico de barras ---
colores = ['#2ecc71', '#e74c3c']
etiquetas = ['Clase 0\nNO Bancarrota\n(6,599 empresas)', 'Clase 1\nEN Bancarrota\n(220 empresas)']
bars = axes[0].bar(etiquetas, [conteo[0], conteo[1]], color=colores, edgecolor='black', linewidth=1.2, width=0.5)

# Añadir porcentajes sobre las barras
for bar, pct in zip(bars, [porcentaje[0], porcentaje[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=13)

axes[0].set_title('Distribución de Clases (Conteo)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Número de Empresas')
axes[0].set_ylim(0, 7500)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# --- Gráfico de torta ---
wedge_props = dict(width=0.5, edgecolor='white', linewidth=3)
axes[1].pie(
    [conteo[0], conteo[1]],
    labels=['NO Bancarrota\n96.77%', 'EN Bancarrota\n3.23%'],
    colors=colores,
    startangle=90,
    wedgeprops=wedge_props,
    textprops={'fontsize': 12}
)
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold', pad=15)

plt.suptitle('Desbalance de Clases en el Dataset de Bancarrota', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print('📌 INTERPRETACIÓN:')
print('   El desbalance 97:3 es un desafío real. Un modelo "tonto" que predice')
print('   SIEMPRE clase 0 tendría 96.77% de accuracy... pero sería inútil.')
print('   Por eso usaremos métricas como Recall y F1-score, no solo Accuracy.')

In [ ]:
# ============================================================
# ESTADÍSTICAS DESCRIPTIVAS
# ------------------------------------------------------------
# Comparamos las empresas en bancarrota vs las sanas
# para entender qué ratios difieren más entre grupos.
# ============================================================

# Separar por clase
sanas     = df[df['Bankrupt?'] == 0]
quebradas = df[df['Bankrupt?'] == 1]

# Seleccionar 10 ratios clave para comparar
ratios_clave = [
    'ROA(C) before interest and depreciation before interest',
    'Debt ratio %',
    'Net worth/Assets',
    'Current Ratio',
    'Quick Ratio',
    'Cash Flow to Sales',
    'Operating Profit Rate',
    'Retained Earnings to Total Assets',
    'Net Income to Total Assets',
    'Borrowing dependency'
]

# Tabla comparativa
comparacion = pd.DataFrame({
    'NO Bancarrota (media)': sanas[ratios_clave].mean(),
    'EN Bancarrota (media)': quebradas[ratios_clave].mean(),
})
comparacion['Diferencia'] = comparacion['NO Bancarrota (media)'] - comparacion['EN Bancarrota (media)']
comparacion['Diferencia %'] = ((comparacion['Diferencia'] / (comparacion['NO Bancarrota (media)'].abs() + 1e-9)) * 100).round(1)

print('COMPARACIÓN DE RATIOS FINANCIEROS: Empresas Sanas vs En Bancarrota')
print('=' * 80)
print(comparacion.round(4).to_string())
print()
print('📌 Ya se observan patrones: las empresas en bancarrota tienden a tener')
print('   menor rentabilidad (ROA), menor liquidez (Current/Quick Ratio)')
print('   y mayor dependencia de deuda (Debt ratio, Borrowing dependency).')

---
## 📚 Sección 2: Análisis Financiero Conceptual

Antes de construir modelos, necesitamos entender qué significan los datos que estamos analizando. Los **ratios financieros** son la materia prima de este proyecto.

---

### ¿Qué son los Ratios Financieros?

Un ratio financiero es una relación matemática entre dos valores contables de una empresa que nos permite evaluar su **salud económica** sin importar el tamaño de la empresa.

Por ejemplo: si una empresa tiene $1,000 de activos y $500 de deuda, su ratio Deuda/Activos es 0.50 (50%). Este número nos dice que la mitad de los activos se financian con deuda.

---

### Categorías de Ratios en el Dataset

| Categoría | ¿Qué mide? | Ejemplo | Señal de riesgo |
|-----------|-----------|---------|----------------|
| **Rentabilidad** | Qué tan bien genera ganancias | ROA, Operating Profit Rate | Valores negativos o muy bajos |
| **Liquidez** | Capacidad de pagar deudas a corto plazo | Current Ratio, Quick Ratio | Ratio < 1.0 |
| **Endeudamiento** | Nivel de deuda respecto a activos | Debt ratio %, Borrowing dependency | Ratios muy altos |
| **Actividad** | Eficiencia operativa | Asset Turnover, Inventory Turnover | Bajos en relación al sector |
| **Flujo de caja** | Dinero real generado | Cash Flow to Sales, CFO to Assets | Flujo negativo |
| **Crecimiento** | Expansión o contracción | Net Value Growth Rate | Crecimiento negativo sostenido |

---

### Patrones Típicos de Empresas en Bancarrota

Según la literatura financiera (Altman, 1968 y estudios posteriores), las empresas que quiebran suelen mostrar:

1. **ROA negativo o muy bajo:** No generan suficiente retorno sobre sus activos.
2. **Current Ratio < 1:** Sus pasivos corrientes superan sus activos corrientes → no pueden pagar deudas inmediatas.
3. **Debt ratio alto (>70%):** Más del 70% de sus activos son financiados con deuda.
4. **Retained Earnings negativos:** Han acumulado pérdidas a lo largo del tiempo.
5. **Flujo de caja negativo:** Gastan más efectivo del que generan.

> 💡 **Contexto histórico:** El modelo **Z-Score de Altman (1968)** fue el primer modelo estadístico de predicción de bancarrota. Usaba 5 ratios financieros. Hoy, con Machine Learning, podemos usar 95 ratios simultáneamente y detectar patrones mucho más complejos.

---
## 🔧 Sección 3: Preprocesamiento de Datos

El preprocesamiento es la etapa donde preparamos los datos para que los algoritmos de ML puedan trabajar con ellos correctamente.

> 🎯 **Regla de oro:** "Garbage in, garbage out" — si los datos no están bien preparados, el modelo no funcionará bien, sin importar qué algoritmo uses.

In [ ]:
# ============================================================
# PASO 1: SEPARAR VARIABLES PREDICTORAS (X) Y OBJETIVO (y)
# ------------------------------------------------------------
# X = las 95 variables de entrada (ratios financieros)
# y = la variable que queremos predecir (Bankrupt?)
# ============================================================

# Variable objetivo
y = df['Bankrupt?']

# Variables predictoras (todo excepto la columna objetivo)
X = df.drop(columns=['Bankrupt?'])

print('SEPARACIÓN DE VARIABLES')
print(f'  X (predictoras) : {X.shape[1]} columnas × {X.shape[0]:,} filas')
print(f'  y (objetivo)    : {y.shape[0]:,} valores  |  clases: {sorted(y.unique())}')
print()

In [ ]:
# ============================================================
# PASO 2: DIVISIÓN TRAIN / TEST
# ------------------------------------------------------------
# Dividimos los datos en:
#   - Train (80%): para entrenar el modelo
#   - Test  (20%): para evaluar qué tan bien generaliza
#
# ¿Por qué dividir?
# Si evaluamos el modelo con los mismos datos que usamos para
# entrenarlo, el modelo 'memoriza' en lugar de 'aprender'.
# El test set simula empresas NUEVAS que el modelo nunca vio.
#
# stratify=y → Mantiene la misma proporción de clases (96%/4%)
# en ambos conjuntos. CRUCIAL con datos desbalanceados.
# random_state=42 → Para reproducibilidad (siempre el mismo resultado)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # ← Muy importante con datos desbalanceados
)

print('DIVISIÓN TRAIN / TEST')
print(f'  Train : {X_train.shape[0]:,} empresas  ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'  Test  : {X_test.shape[0]:,} empresas   ({X_test.shape[0]/len(X)*100:.0f}%)')
print()
print('  Distribución en Train:')
print(f'    Clase 0: {(y_train==0).sum():,}  |  Clase 1: {(y_train==1).sum()}')
print('  Distribución en Test:')
print(f'    Clase 0: {(y_test==0).sum():,}   |  Clase 1: {(y_test==1).sum()}')

In [ ]:
# ============================================================
# PASO 3: ESCALADO DE VARIABLES (StandardScaler)
# ------------------------------------------------------------
# ¿Por qué escalar?
#
# Problema: los ratios tienen escalas MUY diferentes.
# Ejemplo:
#   - 'Revenue per person' puede valer millones (1,000,000)
#   - 'Current Ratio' vale entre 0 y 5
#
# La Regresión Logística asigna PESOS (coeficientes) a cada
# variable. Si una variable tiene valores 1,000,000 veces
# mayores que otra, su coeficiente dominará el modelo aunque
# no sea la más IMPORTANTE.
#
# StandardScaler transforma cada variable para que tenga:
#   - Media = 0
#   - Desviación estándar = 1
#
# REGLA CRUCIAL:
#   fit_transform() en Train → aprende la escala del train
#   transform()     en Test  → aplica la misma escala (no aprende)
#   Así evitamos 'data leakage' (filtración de información del test)
#
# Nota: El Árbol de Decisión NO necesita escalado (no usa distancias),
# pero lo aplicamos igual para mantener consistencia.
# ============================================================

scaler = StandardScaler()

# Ajustar en train y transformar
X_train_scaled = scaler.fit_transform(X_train)

# Solo transformar en test (usando los parámetros aprendidos del train)
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrames para mantener los nombres de columnas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('ESCALADO CON STANDARDSCALER')
print(f'  Antes del escalado — "Revenue per person":')
print(f'    Min: {X_train["Revenue per person"].min():,.0f}  |  Max: {X_train["Revenue per person"].max():,.0f}')
print(f'  Después del escalado — "Revenue per person":')
print(f'    Min: {X_train_scaled["Revenue per person"].min():.3f}  |  Max: {X_train_scaled["Revenue per person"].max():.3f}')
print()
print('✅ Todas las variables ahora tienen media ≈ 0 y desviación estándar ≈ 1')
print()
print('RESUMEN DEL PREPROCESAMIENTO:')
print('  ✅ Valores nulos: Ninguno (no se requirió imputación)')
print('  ✅ Separación X/y: Completada')
print('  ✅ Train/Test split 80/20 con stratify: Completado')
print('  ✅ Escalado StandardScaler: Completado')
print('  ⚠️  Desbalance de clases: se manejará con class_weight="balanced"')

---
## 🤖 Sección 4: Implementación de Modelos

Vamos a implementar y entender en profundidad cada modelo.

---

### 4.1 Regresión Logística

#### ¿Cómo funciona matemáticamente?

A pesar del nombre, la Regresión Logística es un **clasificador**, no un regresor. Su objetivo es calcular la **probabilidad** de que una empresa esté en bancarrota.

**Paso 1 — Combinación lineal de los ratios:**

$$z = b_0 + b_1 x_1 + b_2 x_2 + \cdots + b_n x_n$$

Donde:
- $b_0$ = intercepto (valor base)
- $b_1, b_2, \ldots, b_n$ = coeficientes (pesos de cada ratio)
- $x_1, x_2, \ldots, x_n$ = valores de los ratios financieros

**Paso 2 — Función Logística (Sigmoide):**

$$P(Y=1) = \frac{1}{1 + e^{-z}}$$

Esta función transforma cualquier valor de $z$ (de $-\infty$ a $+\infty$) en una probabilidad entre 0 y 1.

**Paso 3 — Decisión:**
- Si $P(Y=1) \geq 0.5$ → predice **clase 1** (bancarrota)
- Si $P(Y=1) < 0.5$ → predice **clase 0** (sin bancarrota)

**¿Qué significan los coeficientes en finanzas?**
- Coeficiente **positivo** → ese ratio **aumenta** el riesgo de bancarrota
- Coeficiente **negativo** → ese ratio **reduce** el riesgo de bancarrota
- Mayor valor absoluto → mayor influencia sobre la predicción

In [ ]:
# ============================================================
# VISUALIZACIÓN: FUNCIÓN SIGMOIDE
# ------------------------------------------------------------
# Mostramos cómo la función logística convierte z en probabilidad
# ============================================================

z = np.linspace(-10, 10, 300)
sigmoide = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(z, sigmoide, color='#2c3e50', linewidth=2.5, label='Función Sigmoide')

# Zona de decisión
ax.axhline(0.5, color='#e74c3c', linestyle='--', linewidth=1.5, label='Umbral de decisión (0.5)')
ax.axvline(0, color='gray', linestyle=':', linewidth=1)

# Áreas coloreadas
ax.fill_between(z, 0, sigmoide, where=(sigmoide >= 0.5), alpha=0.15, color='#e74c3c', label='Predice: BANCARROTA')
ax.fill_between(z, 0, sigmoide, where=(sigmoide < 0.5),  alpha=0.15, color='#2ecc71', label='Predice: SIN BANCARROTA')

# Anotaciones
ax.annotate('z muy negativo\n(empresa muy sana)', xy=(-7, 0.02), fontsize=10, color='#27ae60', fontweight='bold')
ax.annotate('z muy positivo\n(alto riesgo quiebra)', xy=(4, 0.92), fontsize=10, color='#c0392b', fontweight='bold')
ax.annotate('Zona de\nincertidumbre', xy=(0.2, 0.48), fontsize=9, color='gray')

ax.set_xlabel('z = b₀ + b₁x₁ + b₂x₂ + ... + bₙxₙ  (combinación lineal de ratios)', fontsize=11)
ax.set_ylabel('P(Bancarrota)', fontsize=11)
ax.set_title('Función Sigmoide: cómo la Regresión Logística calcula probabilidades', fontsize=13, fontweight='bold')
ax.legend(loc='center left', fontsize=10)
ax.set_ylim(-0.05, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ENTRENAMIENTO: REGRESIÓN LOGÍSTICA
# ------------------------------------------------------------
# Parámetros clave:
#
# class_weight='balanced':
#   Le dice al modelo que las empresas en bancarrota (clase 1)
#   son MÁS importantes a pesar de ser pocas.
#   Internamente, el modelo multiplica el error de la clase
#   minoritaria por un factor = n_total / (n_clases × n_clase_1)
#   ≈ 6819 / (2 × 220) ≈ 15x más penalización por errores en clase 1
#
# max_iter=1000:
#   Número de iteraciones para el algoritmo de optimización
#   (por defecto 100 puede no ser suficiente con 95 variables)
#
# random_state=42:
#   Para reproducibilidad
# ============================================================

log_reg = LogisticRegression(
    class_weight='balanced',   # Maneja el desbalance
    max_iter=1000,             # Suficientes iteraciones para converger
    random_state=42
)

# Entrenamiento (fit = aprender los coeficientes b0, b1, ..., bn)
log_reg.fit(X_train_scaled, y_train)

# Predicciones en test
y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]  # Probabilidad de clase 1

print('✅ Regresión Logística entrenada')
print(f'   Coeficientes aprendidos  : {len(log_reg.coef_[0])} (uno por variable)')
print(f'   Intercepto (b₀)          : {log_reg.intercept_[0]:.4f}')
print()

# Ejemplo de cómo funciona la predicción
ejemplo_prob = y_prob_lr[0]
ejemplo_pred = y_pred_lr[0]
print(f'   Ejemplo — primera empresa del test:')
print(f'     Probabilidad de bancarrota : {ejemplo_prob:.4f} ({ejemplo_prob*100:.2f}%)')
print(f'     Predicción del modelo      : {"🔴 BANCARROTA" if ejemplo_pred == 1 else "🟢 SIN BANCARROTA"}')
print(f'     Clase real                 : {"🔴 BANCARROTA" if y_test.iloc[0] == 1 else "🟢 SIN BANCARROTA"}')

---
### 4.2 Árbol de Decisión

#### ¿Cómo funciona?

Un Árbol de Decisión imita la forma en que un analista financiero **hace preguntas secuenciales** para evaluar el riesgo de una empresa.

**Ejemplo de lógica de un árbol:**
```
¿El ROA < 0.3?
├── SÍ → ¿El Debt ratio > 0.7?
│         ├── SÍ → 🔴 PREDICE: BANCARROTA
│         └── NO → ¿Current Ratio < 1.0?
│                   ├── SÍ → 🔴 PREDICE: BANCARROTA  
│                   └── NO → 🟢 PREDICE: SIN BANCARROTA
└── NO → 🟢 PREDICE: SIN BANCARROTA
```

**Conceptos clave:**
- **Nodo raíz:** La primera pregunta (el ratio más informativo)
- **Nodo interno:** Una pregunta sobre un ratio financiero
- **Hoja (nodo terminal):** La decisión final (0 o 1)
- **Impureza Gini:** Mide qué tan "mezcladas" están las clases en un nodo. El árbol busca minimizarla.
- **max_depth:** Controla la complejidad del árbol (evita overfitting)

> 💡 **Ventaja en finanzas:** El árbol genera **reglas interpretables**: "Si el ratio de deuda supera X y el ROA es menor que Y, entonces la empresa tiene riesgo de quiebra". Esto es valioso para explicar decisiones a directivos o reguladores.

In [ ]:
# ============================================================
# ENTRENAMIENTO: ÁRBOL DE DECISIÓN
# ------------------------------------------------------------
# Parámetros clave:
#
# max_depth=5:
#   Limita la profundidad del árbol a 5 niveles.
#   Sin límite, el árbol memorizaría los datos de entrenamiento
#   (overfitting) y fallaría con datos nuevos.
#   Con 5 niveles obtenemos reglas interpretables.
#
# min_samples_leaf=10:
#   Cada hoja debe tener al menos 10 empresas.
#   Evita crear reglas basadas en casos muy específicos.
#
# class_weight='balanced':
#   Igual que en regresión logística: compensa el desbalance.
#
# criterion='gini':
#   Usa el índice Gini para medir impureza en cada nodo.
# ============================================================

arbol = DecisionTreeClassifier(
    max_depth=5,              # Árbol interpretable (no muy profundo)
    min_samples_leaf=10,      # Mínimo de empresas por hoja
    class_weight='balanced',  # Maneja el desbalance
    criterion='gini',         # Índice de impureza
    random_state=42
)

# Nota: El árbol NO necesita escalado, pero usamos X_train_scaled
# para mantener consistencia. El árbol solo usa puntos de corte,
# no distancias, así que el escalado no afecta sus resultados.
arbol.fit(X_train_scaled, y_train)

# Predicciones
y_pred_dt = arbol.predict(X_test_scaled)
y_prob_dt = arbol.predict_proba(X_test_scaled)[:, 1]

print('✅ Árbol de Decisión entrenado')
print(f'   Profundidad real del árbol : {arbol.get_depth()}')
print(f'   Número de hojas            : {arbol.get_n_leaves()}')
print(f'   Variables usadas           : {(arbol.feature_importances_ > 0).sum()}')
print()
print('REGLAS DEL ÁRBOL (primeros 3 niveles):')
print('=' * 60)
# Mostrar las reglas en texto
reglas = export_text(arbol, feature_names=list(X.columns), max_depth=3)
# Acortar nombres muy largos para visualización
print(reglas[:3000])

In [ ]:
# ============================================================
# VISUALIZACIÓN DEL ÁRBOL DE DECISIÓN
# ------------------------------------------------------------
# Visualizamos los primeros 3 niveles para que sea legible.
# El color rojo indica mayoría clase 1 (bancarrota),
# el color verde indica mayoría clase 0 (sin bancarrota).
# ============================================================

# Acortar nombres de columnas para la visualización
nombres_cortos = [col[:25] + '...' if len(col) > 25 else col for col in X.columns]

fig, ax = plt.subplots(figsize=(22, 10))

plot_tree(
    arbol,
    max_depth=3,                    # Solo primeros 3 niveles para legibilidad
    feature_names=nombres_cortos,
    class_names=['Sin Quiebra', 'Quiebra'],
    filled=True,                    # Colorear según la clase mayoritaria
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=True,
    proportion=False
)

ax.set_title(
    'Árbol de Decisión para Predicción de Bancarrota\n'
    '(Azul = Sin Quiebra | Naranja = Quiebra | Intensidad = Certeza)',
    fontsize=14, fontweight='bold', pad=15
)

plt.tight_layout()
plt.show()

print()
print('📌 CÓMO LEER EL ÁRBOL:')
print('   • En cada nodo se muestra: el ratio y el punto de corte')
print('   • "samples" = cuántas empresas llegan a ese nodo')
print('   • "value" = [empresas clase 0, empresas clase 1]')
print('   • "class" = la predicción de ese nodo')
print('   • Intensidad del color = certeza de la predicción')

In [ ]:
# ============================================================
# RANDOM FOREST (opcional, para comparar importancia)
# ------------------------------------------------------------
# Random Forest = ensemble de muchos Árboles de Decisión.
#
# ¿Por qué es útil aquí?
# La importancia de variables de un solo árbol puede ser inestable.
# Random Forest promedia la importancia de 200 árboles,
# dando una estimación MUCHO más robusta y confiable de
# qué ratios financieros son realmente importantes.
#
# n_estimators=200: Número de árboles en el bosque
# max_depth=10: Cada árbol puede ser más profundo que el árbol simple
# ============================================================

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1           # Usar todos los núcleos disponibles
)

rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf  = rf.predict_proba(X_test_scaled)[:, 1]

print('✅ Random Forest entrenado')
print(f'   Árboles en el bosque : {rf.n_estimators}')
print(f'   Profundidad máxima   : {rf.max_depth}')

---
## 📏 Sección 5: Evaluación de Modelos

### ¿Por qué no basta con el Accuracy?

Imagina un modelo que **siempre predice "NO bancarrota"**. Tendría un 96.77% de accuracy. ¿Es un buen modelo? **No, es inútil.** Por eso necesitamos métricas más sofisticadas.

---

### La Matriz de Confusión

```
                    PREDICCIÓN DEL MODELO
                   NO Bancarrota    Bancarrota
REALIDAD  NO Banc  Verdadero Neg.   Falso Positivo  ← (alarma falsa)
          Bancarr  Falso Negativo   Verdadero Pos.  ← (detectado)
                       ↑
                   (empresa quebró y no se detectó → el más costoso)
```

### Métricas y su significado financiero

| Métrica | Fórmula | En el contexto de bancarrota |
|---------|---------|-----------------------------|
| **Accuracy** | (VP + VN) / Total | % de predicciones correctas (engañoso con desbalance) |
| **Precision** | VP / (VP + FP) | De las que predije como quiebra, ¿cuántas realmente quebraron? |
| **Recall** | VP / (VP + FN) | De las que quebraron, ¿cuántas detecté? (**la más crítica**) |
| **F1-Score** | 2×(P×R)/(P+R) | Balance entre Precision y Recall |
| **ROC-AUC** | Área bajo la curva ROC | Capacidad general de discriminación |

> 🚨 **Recall es la métrica más importante aquí:**  
> Un **Falso Negativo** (empresa que quiebra pero el modelo no detecta) es devastador:  
> un banco podría seguir prestando dinero a esa empresa,  
> un proveedor seguiría dando crédito,  
> un inversor no retiraría su capital a tiempo.

In [ ]:
# ============================================================
# FUNCIÓN DE EVALUACIÓN COMPLETA
# ------------------------------------------------------------
# Creamos una función reutilizable que evalúa cualquier modelo
# ============================================================

def evaluar_modelo(nombre, y_true, y_pred, y_prob):
    """
    Evalúa un modelo de clasificación y muestra todas las métricas
    en el contexto de predicción de bancarrota.
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)

    vn, fp, fn, vp = cm.ravel()

    print(f'\n{"=" * 65}')
    print(f'  RESULTADOS: {nombre}')
    print(f'{"=" * 65}')
    print(f'  Accuracy    : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision   : {prec:.4f}  — De las predichas como quiebra, {prec*100:.1f}% sí quebraron')
    print(f'  Recall      : {rec:.4f}  — Del total de quiebras reales, detectó el {rec*100:.1f}%  ← MÁS CRÍTICO')
    print(f'  F1-Score    : {f1:.4f}  — Balance Precision/Recall')
    print(f'  ROC-AUC     : {auc:.4f}  — Capacidad discriminativa general')
    print(f'\n  MATRIZ DE CONFUSIÓN:')
    print(f'    Verdaderos Negativos (VN): {vn:4d}  → Sanas identificadas correctamente')
    print(f'    Falsos Positivos    (FP): {fp:4d}  → Sanas identificadas como quiebra (alarma falsa)')
    print(f'    Falsos Negativos    (FN): {fn:4d}  → ⚠️  QUIEBRAS NO DETECTADAS (más peligroso)')
    print(f'    Verdaderos Positivos(VP): {vp:4d}  → Quiebras detectadas correctamente')

    return {'nombre': nombre, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'auc': auc, 'cm': cm}


# Evaluar los tres modelos
res_lr = evaluar_modelo('Regresión Logística', y_test, y_pred_lr, y_prob_lr)
res_dt = evaluar_modelo('Árbol de Decisión',   y_test, y_pred_dt, y_prob_dt)
res_rf = evaluar_modelo('Random Forest',        y_test, y_pred_rf, y_prob_rf)

In [ ]:
# ============================================================
# TABLA COMPARATIVA DE MODELOS
# ============================================================

resultados = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'cm'}
    for r in [res_lr, res_dt, res_rf]
]).set_index('nombre')

print('COMPARACIÓN FINAL DE MODELOS')
print('=' * 65)
print(resultados.round(4).to_string())
print()
print('📌 INTERPRETACIÓN:')
print(f'   Mejor Recall  : {resultados["recall"].idxmax()}  ({resultados["recall"].max():.4f})')
print(f'   Mejor F1      : {resultados["f1"].idxmax()}  ({resultados["f1"].max():.4f})')
print(f'   Mejor AUC     : {resultados["auc"].idxmax()}  ({resultados["auc"].max():.4f})')

In [ ]:
# ============================================================
# VISUALIZACIÓN: MATRICES DE CONFUSIÓN
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
modelos_info = [
    ('Regresión Logística', res_lr['cm'], '#3498db'),
    ('Árbol de Decisión',   res_dt['cm'], '#2ecc71'),
    ('Random Forest',        res_rf['cm'], '#9b59b6'),
]

for ax, (nombre, cm, color) in zip(axes, modelos_info):
    # Calcular porcentajes
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    # Etiquetas combinadas: conteo y porcentaje
    labels = np.array([[f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)'
                        for j in range(2)] for i in range(2)])

    sns.heatmap(
        cm, annot=labels, fmt='', cmap='Blues',
        xticklabels=['Sin Quiebra', 'Quiebra'],
        yticklabels=['Sin Quiebra', 'Quiebra'],
        ax=ax, linewidths=1, linecolor='gray',
        annot_kws={'size': 12, 'weight': 'bold'}
    )
    ax.set_title(f'{nombre}\nF1={res_lr["f1"]:.3f} | Recall={res_lr["recall"]:.3f}' if nombre=='Regresión Logística'
                 else f'{nombre}\nF1={res_dt["f1"]:.3f} | Recall={res_dt["recall"]:.3f}' if nombre=='Árbol de Decisión'
                 else f'{nombre}\nF1={res_rf["f1"]:.3f} | Recall={res_rf["recall"]:.3f}',
                 fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Predicción del Modelo', fontsize=10)
    ax.set_ylabel('Clase Real', fontsize=10)

plt.suptitle('Matrices de Confusión — Comparación de Modelos\n'
             'Fila 0 = Sin Quiebra real  |  Fila 1 = Quiebra real',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print()
print('📌 CÓMO LEER LA MATRIZ:')
print('   Celda [0,0]: Empresas sanas correctamente identificadas ✅')
print('   Celda [0,1]: Empresas sanas predichas como quiebra (alarma falsa) ⚠️')
print('   Celda [1,0]: Quiebras NO detectadas (Falsos Negativos) 🚨 ← EL MAYOR RIESGO')
print('   Celda [1,1]: Quiebras correctamente detectadas ✅')

---
## 🔍 Sección 6: Interpretabilidad — ¿Qué aprendieron los modelos?

Esta es la sección más valiosa desde el punto de vista financiero.

> 💼 **¿Por qué importa la interpretabilidad?**  
> Un banco que usa un modelo de ML para aprobar créditos **debe poder explicar** por qué negó un préstamo. Un modelo que dice "la empresa tiene riesgo" sin explicar por qué no es útil en la práctica. La interpretabilidad cierra la brecha entre el ML y la toma de decisiones empresariales.

Analizaremos:
1. **Coeficientes de la Regresión Logística** → qué ratios aumentan o disminuyen el riesgo
2. **Importancia de variables del Árbol** → qué ratios el árbol usa más para dividir
3. **Importancia del Random Forest** → estimación más robusta

In [ ]:
# ============================================================
# INTERPRETABILIDAD: REGRESIÓN LOGÍSTICA
# ------------------------------------------------------------
# Los coeficientes de la regresión logística representan el
# cambio en el LOG-ODDS de bancarrota por cada unidad
# de cambio (estandarizada) en esa variable.
#
# Como los datos están escalados (StandardScaler),
# los coeficientes SON comparables entre sí.
#
# Coeficiente > 0: ese ratio AUMENTA el riesgo de bancarrota
# Coeficiente < 0: ese ratio REDUCE el riesgo de bancarrota
# ============================================================

# Crear DataFrame con coeficientes
coefs_lr = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': log_reg.coef_[0]
}).sort_values('Coeficiente', key=abs, ascending=False)

# Top 15 más influyentes
top_lr = coefs_lr.head(15)

print('TOP 15 RATIOS MÁS INFLUYENTES — REGRESIÓN LOGÍSTICA')
print('=' * 65)
print(f'{"Variable":<50} {"Coeficiente":>12}  {"Efecto"}')
print('-' * 65)
for _, row in top_lr.iterrows():
    efecto = '🔴 AUMENTA riesgo' if row['Coeficiente'] > 0 else '🟢 REDUCE  riesgo'
    print(f"{row['Variable'][:49]:<50} {row['Coeficiente']:>12.4f}  {efecto}")

print()
print('📌 INTERPRETACIÓN FINANCIERA:')
print('   Los ratios con coeficiente MÁS NEGATIVO son los que más protegen')
print('   contra la bancarrota (rentabilidad, liquidez, cobertura).')
print('   Los ratios con coeficiente MÁS POSITIVO son señales de alerta')
print('   (endeudamiento excesivo, flujo de caja negativo).')

In [ ]:
# ============================================================
# VISUALIZACIÓN: COEFICIENTES REGRESIÓN LOGÍSTICA
# ============================================================

fig, ax = plt.subplots(figsize=(13, 8))

# Tomar top 20 por valor absoluto
top20_lr = coefs_lr.head(20).sort_values('Coeficiente')
nombres_cortos = [n[:40] + '...' if len(n) > 40 else n for n in top20_lr['Variable']]

colores_coef = ['#e74c3c' if c > 0 else '#2ecc71' for c in top20_lr['Coeficiente']]

bars = ax.barh(nombres_cortos, top20_lr['Coeficiente'], color=colores_coef,
               edgecolor='black', linewidth=0.5, height=0.7)

ax.axvline(0, color='black', linewidth=1.5)
ax.set_xlabel('Coeficiente (escalado)', fontsize=12)
ax.set_title('Top 20 Ratios Financieros — Regresión Logística\n'
             'Rojo = Aumenta riesgo de quiebra  |  Verde = Reduce riesgo de quiebra',
             fontsize=13, fontweight='bold', pad=15)

# Leyenda
patch_r = mpatches.Patch(color='#e74c3c', label='Aumenta riesgo de bancarrota')
patch_g = mpatches.Patch(color='#2ecc71', label='Reduce riesgo de bancarrota')
ax.legend(handles=[patch_r, patch_g], loc='lower right', fontsize=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# IMPORTANCIA DE VARIABLES: ÁRBOL vs RANDOM FOREST
# ------------------------------------------------------------
# La importancia de variables en árboles se basa en cuánto
# REDUCE la impureza Gini cada variable en todos los nodos
# donde se usa. Una variable muy usada en nodos tempranos
# (más arriba del árbol) tiene mayor importancia.
# ============================================================

# Importancia del Árbol
imp_dt = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_DT': arbol.feature_importances_
}).sort_values('Importancia_DT', ascending=False)

# Importancia del Random Forest
imp_rf = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_RF': rf.feature_importances_
}).sort_values('Importancia_RF', ascending=False)

# Combinar para comparar
imp_merged = imp_dt.merge(imp_rf, on='Variable')
imp_merged['Rank_DT'] = imp_merged['Importancia_DT'].rank(ascending=False).astype(int)
imp_merged['Rank_RF'] = imp_merged['Importancia_RF'].rank(ascending=False).astype(int)
imp_merged = imp_merged.sort_values('Importancia_RF', ascending=False)

print('TOP 15 RATIOS MÁS IMPORTANTES')
print('=' * 80)
print(f'{"Variable":<48} {"Árbol":>10} {"R.Forest":>10} {"Rank DT":>8} {"Rank RF":>8}')
print('-' * 80)
for _, row in imp_merged.head(15).iterrows():
    print(f"{row['Variable'][:47]:<48} {row['Importancia_DT']:>10.4f} "
          f"{row['Importancia_RF']:>10.4f} {row['Rank_DT']:>8} {row['Rank_RF']:>8}")

print()
print('📌 NOTA: El Random Forest da estimaciones más estables porque')
print('   promedia la importancia de 200 árboles entrenados con')
print('   subconjuntos aleatorios de datos y variables.')

In [ ]:
# ============================================================
# VISUALIZACIÓN: IMPORTANCIA DE VARIABLES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Árbol de Decisión ---
top15_dt = imp_dt.head(15).sort_values('Importancia_DT')
nombres_dt = [n[:38] + '...' if len(n) > 38 else n for n in top15_dt['Variable']]
axes[0].barh(nombres_dt, top15_dt['Importancia_DT'],
             color='#3498db', edgecolor='black', linewidth=0.5)
axes[0].set_title('Importancia de Variables\nÁrbol de Decisión', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Reducción de Impureza Gini')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# --- Random Forest ---
top15_rf = imp_rf.head(15).sort_values('Importancia_RF')
nombres_rf = [n[:38] + '...' if len(n) > 38 else n for n in top15_rf['Variable']]
axes[1].barh(nombres_rf, top15_rf['Importancia_RF'],
             color='#9b59b6', edgecolor='black', linewidth=0.5)
axes[1].set_title('Importancia de Variables\nRandom Forest (200 árboles)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Reducción de Impureza Gini (promedio)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('¿Qué ratios financieros predicen mejor la bancarrota?',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print()
print('📌 INTERPRETACIÓN FINANCIERA DE LOS RATIOS MÁS IMPORTANTES:')
print()
print('   Net worth/Assets: Proporción del patrimonio sobre activos totales.')
print('   Mayor valor = empresa más solvente. Empresas en quiebra tienen valores bajos.')
print()
print('   Debt ratio %: Qué porcentaje de los activos se financia con deuda.')
print('   Valores altos (>70%) son señal de peligro.')
print()
print('   ROA (Return on Assets): Rentabilidad por cada unidad de activo.')
print('   ROA negativo o muy bajo es típico de empresas en deterioro.')
print()
print('   Retained Earnings/Total Assets: Ganancias acumuladas históricas.')
print('   Negativo significa que la empresa ha tenido pérdidas sostenidas.')

---
## 📊 Sección 7: Visualizaciones

En esta sección generamos visualizaciones que nos ayudan a entender la estructura del dataset y los resultados desde una perspectiva financiera.

In [ ]:
# ============================================================
# HEATMAP DE CORRELACIÓN
# ------------------------------------------------------------
# Un heatmap de correlación muestra qué tan relacionadas están
# las variables entre sí.
#
# Correlación = 1.0  → perfectamente relacionadas (misma dirección)
# Correlación = -1.0 → perfectamente opuestas
# Correlación = 0.0  → sin relación lineal
#
# En finanzas: si dos ratios están muy correlacionados (>0.9)
# probablemente miden lo mismo. Incluir ambos no agrega información.
# ============================================================

# Seleccionamos los ratios más importantes del Random Forest
top_vars = imp_rf.head(15)['Variable'].tolist()
top_vars_con_target = ['Bankrupt?'] + top_vars

# Calcular correlaciones
corr_matrix = df[top_vars_con_target].corr()

# Nombres cortos para la visualización
nombres_heatmap = ['Bankrupt?'] + [n[:20] for n in top_vars]
corr_matrix_display = corr_matrix.copy()
corr_matrix_display.index   = nombres_heatmap
corr_matrix_display.columns = nombres_heatmap

fig, ax = plt.subplots(figsize=(16, 13))

mask = np.zeros_like(corr_matrix_display, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # Mostrar solo triángulo inferior

sns.heatmap(
    corr_matrix_display,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',     # Rojo = negativa, Verde = positiva
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    annot_kws={'size': 8},
    ax=ax
)

ax.set_title('Heatmap de Correlación — Top 15 Ratios Financieros + Variable Objetivo\n'
             'Verde = Correlación positiva  |  Rojo = Correlación negativa',
             fontsize=13, fontweight='bold', pad=20)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
plt.show()

print()
print('📌 CÓMO LEER EL HEATMAP:')
print('   Primera fila/columna: correlación de cada ratio con Bankrupt?')
print('   Rojo oscuro con Bankrupt?: ese ratio sube cuando hay quiebra')
print('   Verde oscuro con Bankrupt?: ese ratio baja cuando hay quiebra')
print('   Pares con correlación >0.9 entre ratios: probablemente redundantes')

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE RATIOS CLAVE POR CLASE
# ------------------------------------------------------------
# Comparamos la distribución de los 6 ratios más importantes
# entre empresas sanas (clase 0) y en quiebra (clase 1).
# Si las distribuciones se SEPARAN → el ratio es informativo.
# ============================================================

top6_vars = imp_rf.head(6)['Variable'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

colores_clase = {0: '#2ecc71', 1: '#e74c3c'}

for i, var in enumerate(top6_vars):
    for clase, color in colores_clase.items():
        datos = df[df['Bankrupt?'] == clase][var]
        # Eliminar outliers extremos para mejor visualización
        q1, q99 = datos.quantile([0.01, 0.99])
        datos_filtrados = datos[(datos >= q1) & (datos <= q99)]

        etiqueta = 'Sin Bancarrota (0)' if clase == 0 else 'En Bancarrota (1)'
        axes[i].hist(datos_filtrados, bins=40, alpha=0.6, color=color,
                     label=etiqueta, density=True, edgecolor='none')

    nombre_corto = var[:35] + '...' if len(var) > 35 else var
    axes[i].set_title(nombre_corto, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valor del Ratio', fontsize=9)
    axes[i].set_ylabel('Densidad', fontsize=9)
    axes[i].legend(fontsize=8)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

plt.suptitle('Distribución de los 6 Ratios Más Importantes por Clase\n'
             'Si las curvas se separan, el ratio discrimina bien entre sanas y quebradas',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print()
print('📌 INTERPRETACIÓN:')
print('   Cuando la distribución ROJA (quiebra) se desplaza hacia valores')
print('   más altos o más bajos que la VERDE (sana), ese ratio es útil')
print('   para distinguir empresas en riesgo.')

In [ ]:
# ============================================================
# VISUALIZACIÓN: COMPARACIÓN DE MÉTRICAS
# ============================================================

metricas = ['accuracy', 'precision', 'recall', 'f1', 'auc']
nombres_modelos = ['Reg. Logística', 'Árbol de Decisión', 'Random Forest']

valores = np.array([
    [res_lr[m] for m in metricas],
    [res_dt[m] for m in metricas],
    [res_rf[m] for m in metricas],
])

x = np.arange(len(metricas))
ancho = 0.25
colores_modelos = ['#3498db', '#2ecc71', '#9b59b6']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (modelo, color) in enumerate(zip(nombres_modelos, colores_modelos)):
    bars = ax.bar(x + i * ancho, valores[i], ancho, label=modelo,
                  color=color, alpha=0.85, edgecolor='black', linewidth=0.7)
    # Etiquetas de valor
    for bar, val in zip(bars, valores[i]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

# Resaltar Recall (la métrica más importante)
ax.axvspan(x[2] - 0.15, x[2] + 0.9, alpha=0.08, color='red',
           label='Recall: métrica crítica en bancarrota')

ax.set_xticks(x + ancho)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall\n(⬅ MÁS CRÍTICO)', 'F1-Score', 'ROC-AUC'],
                   fontsize=11)
ax.set_ylabel('Valor de la Métrica', fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_title('Comparación de Métricas por Modelo\nContexto: Predicción de Bancarrota Empresarial',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 🎯 Sección 8: Conclusiones

### ¿Qué aprendimos?

In [ ]:
# ============================================================
# RESUMEN EJECUTIVO FINAL
# ============================================================

print('=' * 70)
print('RESUMEN EJECUTIVO — PREDICCIÓN DE BANCARROTA EMPRESARIAL')
print('=' * 70)

print()
print('📊 DATASET')
print(f'   • 6,819 empresas  |  95 ratios financieros  |  Sin valores nulos')
print(f'   • Desbalance severo: 96.77% sanas / 3.23% en quiebra')
print(f'   • Solución al desbalance: class_weight="balanced" en todos los modelos')

print()
print('🏆 RENDIMIENTO DE MODELOS (sobre datos de TEST nunca vistos)')
print(f'   {"Modelo":<25} {"Recall":>8} {"F1":>8} {"AUC":>8}')
print(f'   {"-"*55}')
for res in [res_lr, res_dt, res_rf]:
    print(f'   {res["nombre"]:<25} {res["recall"]:>8.4f} {res["f1"]:>8.4f} {res["auc"]:>8.4f}')

print()
print('🔍 RATIOS FINANCIEROS MÁS PREDICTIVOS (según Random Forest):')
for i, (_, row) in enumerate(imp_rf.head(5).iterrows(), 1):
    print(f'   {i}. {row["Variable"]}')

print()
print('📌 HALLAZGOS FINANCIEROS CLAVE')
print('   • Las empresas en quiebra tienen MENOR patrimonio neto relativo')
print('     (Net worth/Assets bajo = más deuda que capital propio)')
print('   • Mayor Debt ratio % = mayor riesgo (más apalancamiento financiero)')
print('   • ROA negativo o muy bajo = no generan retorno sobre sus activos')
print('   • Retained Earnings negativas = pérdidas acumuladas históricas')
print('   • Baja liquidez (Current Ratio, Quick Ratio) = no pueden pagar deudas inmediatas')

print()
print('📋 INTERPRETABILIDAD')
print('   • Regresión Logística: más interpretable (coeficientes directos)')
print('     → Ideal para explicar a directivos o reguladores')
print('   • Árbol de Decisión: genera reglas financieras claras (if-then)')
print('     → Ideal para sistemas de scoring o checklists de riesgo')
print('   • Random Forest: mejor rendimiento, pero menos interpretable')
print('     → Útil para estimación robusta de importancia de variables')

print()
print('⚠️  LIMITACIONES')
print('   • El dataset es histórico y puede no capturar crisis económicas recientes')
print('   • El desbalance extremo hace difícil detectar TODOS los casos de quiebra')
print('   • Los ratios son de empresas taiwanesas; puede necesitar adaptación regional')
print('   • Un modelo predictivo no reemplaza el juicio humano experto')

print()
print('🌎 APLICACIONES EN LA VIDA REAL')
print('   💳 Banca: Scoring de crédito — decidir si otorgar préstamos')
print('   📈 Inversión: Filtrar empresas de alto riesgo en portafolios')
print('   🏛️  Regulación: Supervisión temprana de empresas en dificultad')
print('   🤝 Proveedores: Evaluar riesgo de crédito comercial')
print('   🔍 Auditoría: Priorizar revisiones de empresas con señales de alerta')

print()
print('=' * 70)
print('FIN DEL PROYECTO')
print('=' * 70)

---

## 📝 Reflexión Final

### ¿Qué modelo elegirías en la práctica?

Depende del contexto:

| Escenario | Modelo recomendado | Razón |
|-----------|-------------------|-------|
| Banco que debe **justificar** la negación de crédito | Regresión Logística | Los coeficientes son auditables y explicables |
| Analista que quiere **reglas simples** para un checklist | Árbol de Decisión | Genera reglas tipo "si X < Y, entonces..." |
| Sistema automático donde prima el **rendimiento** | Random Forest | Mejor detección aunque menos explicable |

### Conexión con la historia del análisis financiero

En 1968, Edward Altman propuso el **Z-Score**, una combinación lineal de 5 ratios financieros para predecir quiebras. Era esencialmente una **regresión lineal discriminante**. Hoy, con Machine Learning y 95 ratios, podemos detectar patrones mucho más complejos. Sin embargo, el principio es el mismo: **los números cuentan la historia de una empresa antes de que esta quiebre**.

---

> 🎓 **Lo más importante que aprendiste en este proyecto:**  
> El Machine Learning en finanzas no se trata de maximizar el accuracy. Se trata de **detectar el riesgo correcto, en el momento correcto, con la explicación adecuada**. Un modelo que detecta el 80% de las quiebras y puede explicar por qué, es infinitamente más valioso que uno que detecta el 85% pero es una caja negra.

---
*Proyecto desarrollado con Python, scikit-learn, pandas, matplotlib y seaborn.*